现在我们已经有模型和数据集，接下来就要在数据上优化模型参数，完成模型的训练、验证与测试。

模型训练是一个迭代过程：每一轮迭代中，模型会对输出做出预测，计算预测的误差（损失 loss）；求出误差相对于各个模型参数的导数（上一节已经讲过），再使用梯度下降对参数做优化。

先把之前的代码（03节）复制过来

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

### PART1.超参数

超参数是可手动调节的参数，用来控制模型的优化过程。不同的超参数取值，会影响模型训练效果以及收敛速度（阅读更多关于超参数调优的内容）。
我们为训练定义如下超参数：
- 训练轮数（Number of Epochs）：完整遍历整个数据集的次数
- 批次大小（Batch Size）：在更新模型参数之前，送入网络进行计算的数据样本数量
- 学习率（Learning Rate）：每一个批次 / 每一轮迭代中，模型参数的更新幅度。学习率过小会导致学习速度很慢；取值过大，则训练过程容易出现不可控的异常表现。

In [2]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

### PART2.优化循环

设置好超参数之后，就可以通过优化循环来训练、优化模型。优化循环的每一次迭代称为一个 epoch（训练轮）。

每一轮 epoch 包含两大组成部分：
- 训练循环（Train Loop）：遍历训练数据集，让模型收敛到最优参数。
- 验证 / 测试循环（Validation/Test Loop）：遍历测试数据集，检验模型性能是否在提升。

下面我们快速熟悉训练循环用到的部分概念。可以直接跳转查看优化循环的完整实现。

### PART3.损失函数

给定训练数据时，未训练的网络大概率无法输出正确结果。损失函数用来衡量模型输出结果与目标真实值之间的差距；训练过程就是要最小化这个损失函数。
计算损失时，使用样本输入得到模型预测结果，再把预测结果和真实标签做对比。

常见损失函数：
- nn.MSELoss（均方误差）：用于回归任务
- nn.NLLLoss（负对数似然损失）：用于分类任务
- nn.CrossEntropyLoss：内部组合了 nn.LogSoftmax 和 nn.NLLLoss

我们直接把模型输出的原始得分 logits 送入 nn.CrossEntropyLoss，该函数会自动对 logits 做归一化，并计算预测误差。

In [ ]:
loss_fn = nn.CrossEntropyLoss()

### PART4.优化器

优化，就是在每一步训练中调整模型参数，降低模型误差的过程。优化算法规定了该如何完成这一过程（本示例使用随机梯度下降）。全部优化逻辑都封装在optimizer（优化器）对象中。这里我们使用 SGD 优化器；此外 PyTorch 还提供许多其他优化器，例如 ADAM、RMSProp，它们在不同模型与数据集上表现效果更好。

初始化优化器时，需要注册模型中待训练的参数，并传入学习率这个超参数。

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

在训练循环内部，优化分为三步执行：
- 调用optimizer.zero_grad()，重置模型参数的梯度。梯度默认会累加；为避免重复计数，每一轮迭代都需要手动把梯度置零。
- 调用loss.backward()，对预测损失执行反向传播。PyTorch 会计算损失相对于每一个参数的梯度并保存下来。
- 得到梯度之后，调用optimizer.step()，根据反向传播得到的梯度更新模型参数。

### PART5.完整实现

我们定义train_loop函数，封装全部优化训练逻辑；再定义test_loop函数，在测试数据集上评估模型性能。

In [3]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # 将模型设置为训练模式 —— 对批归一化层、dropout层至关重要
    # 本示例里其实用不到，但写上属于最佳实践
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # 计算预测值与损失
        pred = model(X)
        loss = loss_fn(pred, y)
        # 反向传播
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # 将模型设置为评估模式 —— 对批归一化层、dropout层至关重要
    # 本示例里其实用不到，但写上属于最佳实践
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # 使用 torch.no_grad() 评估模型，确保测试阶段不会计算梯度
    # 同时避免对 requires_grad=True 的张量做不必要的梯度运算，节省显存
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

我们初始化损失函数与优化器，并将它们传入train_loop和test_loop。你可以自行增大训练轮数，观察模型性能逐步提升的过程。

In [4]:
# 定义损失函数：交叉熵损失，用于分类任务
loss_fn = nn.CrossEntropyLoss()
# 构建SGD随机梯度下降优化器，传入模型参数与学习率超参数
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# 设置训练总轮数
epochs = 10
# 循环执行每一轮训练+测试
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    # 执行训练循环
    train_loop(train_dataloader, model, loss_fn, optimizer)
    # 执行测试/验证循环
    test_loop(test_dataloader, model, loss_fn)
print("完成!")

Epoch 1
-------------------------------
loss: 2.303177  [   64/60000]
loss: 2.283616  [ 6464/60000]
loss: 2.269521  [12864/60000]
loss: 2.263723  [19264/60000]
loss: 2.228878  [25664/60000]
loss: 2.215815  [32064/60000]
loss: 2.216852  [38464/60000]
loss: 2.184843  [44864/60000]
loss: 2.186439  [51264/60000]
loss: 2.147910  [57664/60000]
Test Error: 
 Accuracy: 48.3%, Avg loss: 2.144621 

Epoch 2
-------------------------------
loss: 2.162557  [   64/60000]
loss: 2.148149  [ 6464/60000]
loss: 2.091839  [12864/60000]
loss: 2.100762  [19264/60000]
loss: 2.033406  [25664/60000]
loss: 1.985709  [32064/60000]
loss: 2.008589  [38464/60000]
loss: 1.931694  [44864/60000]
loss: 1.935126  [51264/60000]
loss: 1.848458  [57664/60000]
Test Error: 
 Accuracy: 56.5%, Avg loss: 1.854406 

Epoch 3
-------------------------------
loss: 1.900274  [   64/60000]
loss: 1.864178  [ 6464/60000]
loss: 1.744831  [12864/60000]
loss: 1.779891  [19264/60000]
loss: 1.650736  [25664/60000]
loss: 1.616035  [32064/600